In [4]:
# SDG-Based Interactive Round Robin Visualizer (compact) - paste in ONE Colab cell and run
import io, os, time, threading, zipfile
from collections import deque
import pandas as pd, ipywidgets as w
from IPython.display import display
from matplotlib.figure import Figure

OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
A1 = [("P1","Water-quality monitoring",0,5), ("P2","Waste-collection scheduling",1,3),
      ("P3","Traffic-pollution analysis",2,4), ("P4","Public energy-use monitoring",3,2)]
A2 = [("P1","Flood-warning analysis",0,5), ("P2","Air-quality alert generation",1,4),
      ("P3","Emergency evacuation route update",2,3), ("P4","Renewable-grid status monitoring",3,2)]
ACT = {"Activity 1 - Municipal Services (0 ms overhead)": ("act1", A1, 0, [2, 3]),
       "Activity 2 - Climate Emergency (1 ms overhead)":  ("act2", A2, 1, [2, 4])}
COL = {"P1":"#4C78A8", "P2":"#59A14F", "P3":"#F58518", "P4":"#B279A2", "CS":"#9D9D9D"}

# ---- 1. Round Robin engine: one snapshot S[t] for EVERY simulated millisecond ----
def simulate(procs, quantum, cs):
    pend = sorted(procs, key=lambda p: p[2])             # processes not yet arrived
    rem = {p[0]: p[3] for p in procs}
    Q, S, first, done = deque(), [], {}, {}
    t = last = sw = busy = csms = blk = 0; last = None
    def admit():                                         # arrived processes join the queue
        new = []
        while pend and pend[0][2] <= t: n = pend.pop(0)[0]; Q.append(n); new.append(n)
        return new
    def snap(label, note, new):
        S.append(dict(t=t, label=label, queue=list(Q), rem=dict(rem), done=dict(done), first=dict(first),
                      new=new, note=note, busy=busy, csms=csms, sw=sw, blk=blk))
    snap(None, "Time 0: simulation ready. Arrivals at 0 ms join the ready queue.", admit())
    while len(done) < len(procs):
        if not Q: t += 1; snap(None, "CPU idle.", admit()); continue
        p = Q.popleft(); pre = ""                        # FIFO: take the front process
        if last and last != p:                           # different process -> context switch
            sw += 1; blk += 1; pre = f"(Switch {last}->{p} costs 0 ms, counted as 1 switch.) "
            for _ in range(cs):
                t += 1; csms += 1
                snap("CS", f"CONTEXT SWITCH {last} -> {p}: CPU is doing switching work, no process runs.", admit()); pre = ""
        run = min(quantum, rem[p]); blk += 1; first.setdefault(p, t)
        for k in range(run):
            t += 1; rem[p] -= 1; busy += 1; new = admit(); ev = ""   # arrivals BEFORE re-adding p
            if k == run - 1:
                if rem[p] == 0: done[p] = t; ev = f"{p} FINISHED, CT = {t}."
                else: Q.append(p); ev = f"Quantum over: {p} goes to the back of the queue."
            snap(p, f"{pre if k == 0 else ''}{p} runs 1 ms (remaining {rem[p]}). " +
                 (f"{', '.join(new)} arrived. " if new else "") + ev, new)
        last = p
    return S

def results(procs, S, cs):                               # TAT = CT-AT, WT = TAT-BT, RT = first start - AT
    f = S[-1]; T = f["t"]
    df = pd.DataFrame([(n, a, b, f["done"][n], f["done"][n]-a, f["done"][n]-a-b, f["first"][n]-a)
                       for n, _, a, b in procs], columns=["Process","AT","BT","CT","TAT","WT","RT"])
    sm = {"Avg TAT": round(df.TAT.mean(), 2), "Avg WT": round(df.WT.mean(), 2), "Avg RT": round(df.RT.mean(), 2),
          "Context switches": f["sw"], "CS overhead (ms)": f["sw"]*cs, "Total time (ms)": T,
          "CPU efficiency %": round(100*f["busy"]/T, 2)}
    return df, sm

RUNS = {}
for name, (key, procs, cs, qs) in ACT.items():
    for q in qs:
        S = simulate(procs, q, cs); RUNS[(key, q)] = (S,) + results(procs, S, cs)

# ---- 2. Drawing helpers ----
def gantt(S, procs, upto, title):                        # Gantt chart filled up to time 'upto'
    fig = Figure(figsize=(11, 2.8)); ax = fig.subplots(); B = {}
    for s in S[1:upto+1]:
        b = B.setdefault(s["blk"], [s["t"]-1, s["t"], s["label"]]); b[1] = s["t"]
    for a, b, lab in B.values():
        ax.barh(0, b-a, left=a, height=.6, color=COL[lab], edgecolor="black", hatch="//" if lab == "CS" else None)
        ax.text((a+b)/2, 0, lab, ha="center", va="center", fontweight="bold", color="black" if lab == "CS" else "white")
    for n, _, a, _ in procs:
        if a <= upto: ax.plot(a, .42, "v", color=COL[n], clip_on=False); ax.text(a, .5, n, ha="center", color=COL[n], fontsize=8)
    ax.axvline(upto, color="red", ls="--"); ax.set_xlim(0, S[-1]["t"]); ax.set_ylim(-.45, .85)
    ax.set_xticks(range(S[-1]["t"]+1)); ax.set_yticks([]); ax.grid(axis="x", alpha=.3)
    ax.set_xlabel("Time (ms)   (grey hatched = context switch, red line = now)"); ax.set_title(title, fontweight="bold")
    fig.tight_layout(); buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=100); return buf.getvalue()

CSS = ("<style>.rr table{border-collapse:collapse;font-family:Arial;font-size:13px}.rr th{background:#34495e;color:white;"
       "padding:4px 10px}.rr td{border:1px solid #ccc;padding:3px 10px;text-align:center}"
       ".c{border-radius:8px;padding:8px 12px;margin:4px 0;font-family:Arial;font-size:14px;color:#12345B !important;}"
       ".c *{color:#12345B !important;}</style>")

def card(txt, bg, bd): return f"<div class='c' style='background:{bg};border-left:6px solid {bd}'>{txt}</div>"
def table(df): return f"<div class='rr'>{df.to_html(index=False, border=0)}</div>"

def panel(S, procs, cs, t):                              # all text panels for time t
    s = S[t]; fin = len(s["done"]) == len(procs)
    cpu = ("Not started" if t == 0 else "All processes completed" if fin and t == S[-1]["t"] else
           "<b>CS</b> - CPU is performing a context switch" if s["label"] == "CS" else f"<b>{s['label']}</b> is executing")
    rows = []
    for n, task, a, b in procs:
        st = (f"Completed (CT={s['done'][n]})" if n in s["done"] else "Running" if s["label"] == n else
              "Ready" if n in s["queue"] else "Not arrived" if a > t else "Ready (switching to it)")
        rows.append((n, task, a, b, s["rem"][n], st))
    ptab = pd.DataFrame(rows, columns=["Process","Task","AT","BT","Remaining","Status"])
    waits = [(s["done"].get(n, t) - a) - (b - s["rem"][n]) for n, _, a, b in procs if a <= t]
    live = card(f"<b>LIVE values at {t} ms (change every step)</b><br>CPU busy: {s['busy']} ms | CS time: {s['csms']} ms | "
                f"Switches so far: {s['sw']} | Avg waiting so far: {sum(waits)/len(waits) if waits else 0:.2f} ms | "
                f"Live efficiency: {100*s['busy']/t if t else 0:.1f}%", "#FDEBD0", "#E67E22")
    if fin:
        df, sm = results(procs, S, cs)
        final = card("<b>FINAL values (simulation finished)</b>" + table(df) +
                     " | ".join(f"{k}: <b>{v}</b>" for k, v in sm.items()), "#D5F5E3", "#27AE60")
    else: final = card("<b>FINAL values</b> (CT, TAT, WT, RT, averages, efficiency) appear when the simulation finishes.", "#EEE", "#999")
    top = (card(f"<span style='font-size:24px'><b>Time = {t} ms</b></span> &nbsp; CPU: {cpu}<br>"
                f"<b>Ready queue (front to back):</b> {' -> '.join(s['queue']) or '(empty)'}<br>"
                f"<b>Arrived now:</b> {', '.join(s['new']) or 'none'} | <b>Completed:</b> {', '.join(s['done']) or 'none'}", "#EAF2FB", "#2E86C1"))
    return CSS + top, card("<b>What happens now:</b> " + s["note"], "#FEF9E7", "#F1C40F"), table(ptab), live + final

# ---- 3. Comparison tables + conclusion ----
def comp_df(key, qs): return pd.DataFrame({f"Q = {q} ms": RUNS[(key, q)][2] for q in qs}).rename_axis("Metric").reset_index()

def conclusion(key, qs, cs):
    sa, sb = RUNS[(key, qs[0])][2], RUNS[(key, qs[1])][2]
    b = qs[1] if (sb["Avg WT"], sb["Context switches"]) < (sa["Avg WT"], sa["Context switches"]) else qs[0]
    sm = RUNS[(key, b)][2]; so = RUNS[(key, qs[0] if b == qs[1] else qs[1])][2]
    t = (f"<b>Conclusion:</b> Q = {b} ms performs better: avg WT {sm['Avg WT']} vs {so['Avg WT']} ms, avg TAT {sm['Avg TAT']} vs "
         f"{so['Avg TAT']} ms, switches {sm['Context switches']} vs {so['Context switches']}" +
         (f", overhead {sm['CS overhead (ms)']} vs {so['CS overhead (ms)']} ms, CPU efficiency {sm['CPU efficiency %']}% vs {so['CPU efficiency %']}%" if cs else "") +
         f".<br><b>Trade-off:</b> a smaller quantum gives faster first responses (avg RT {RUNS[(key, qs[0])][2]['Avg RT']} ms for Q={qs[0]} vs "
         f"{RUNS[(key, qs[1])][2]['Avg RT']} ms for Q={qs[1]}) but causes more preemptions and context switches"
         + (" (more wasted overhead time)" if cs else "") + "; a larger quantum switches less but later processes wait longer for their first turn.")
    return card(t, "#E8F8F5", "#1ABC9C")

def save_all():                                          # PNG charts + CSV tables -> ZIP
    files = []
    for name, (key, procs, cs, qs) in ACT.items():
        for q in qs:
            p = f"{OUT}/{key}_gantt_chart_quantum_{q}ms.png"; open(p, "wb").write(gantt(RUNS[(key, q)][0], procs, RUNS[(key, q)][0][-1]["t"], f"{name}, Q={q} ms")); files.append(p)
            p = f"{OUT}/{key}_results_quantum_{q}ms.csv"; RUNS[(key, q)][1].to_csv(p, index=False); files.append(p)
        p = f"{OUT}/{key}_comparison_quantum_{qs[0]}ms_vs_{qs[1]}ms.csv"; comp_df(key, qs).to_csv(p, index=False); files.append(p)
    with zipfile.ZipFile(f"{OUT}/round_robin_results.zip", "w") as z:
        for p in files: z.write(p, os.path.basename(p))
    return f"{OUT}/round_robin_results.zip"

# ---- 4. Interactive interface ----
def build_ui():
    st = dict(name=list(ACT)[0], q=2, t=0, playing=False, token=0, loading=False, mute=False)
    act = w.Dropdown(options=list(ACT), description="Activity:", layout=w.Layout(width="520px"))
    qd = w.Dropdown(options=[2, 3], description="Quantum (ms):")
    speed = w.FloatSlider(value=1.5, min=.5, max=10, step=.5, description="Speed (ms/s):")
    bt = {n: w.Button(description=n, icon=i, button_style=s) for n, i, s in
          [("Play","play","success"), ("Pause","pause","warning"), ("Previous Step","step-backward","info"),
           ("Next Step","step-forward","info"), ("Reset","refresh","danger")]}
    scrub = w.IntSlider(value=0, min=0, max=1, description="Time (ms):", layout=w.Layout(width="95%"))
    top, expl, ptab, mets = w.HTML(), w.HTML(), w.HTML(), w.HTML(); img = w.Image(format="png", value=b"", layout=w.Layout(width="95%"))
    lock = threading.Lock()
    def cur(): key, procs, cs, qs = ACT[st["name"]]; return key, procs, cs, RUNS[(key, st["q"])][0]

    def render():                                        # redraw everything for the current time
        with lock:
            key, procs, cs, S = cur(); t = st["t"]
            top.value, expl.value, ptab.value, mets.value = panel(S, procs, cs, t)
            img.value = gantt(S, procs, t, f"Live Gantt - {st['name']}, Q = {st['q']} ms")
            st["mute"] = True; scrub.max = S[-1]["t"]; scrub.value = t; st["mute"] = False
            bt["Previous Step"].disabled = t == 0; bt["Next Step"].disabled = t >= S[-1]["t"]
            bt["Play"].disabled = st["playing"]; bt["Pause"].disabled = not st["playing"]
    def pause(_=None): st["playing"] = False; st["token"] += 1; bt["Play"].disabled = False; bt["Pause"].disabled = True
    def player(tok):                                     # background loop: exactly 1 ms per step
        while True:
            time.sleep(1 / speed.value)
            if not st["playing"] or st["token"] != tok: return
            st["t"] += 1; render()
            if st["t"] >= cur()[3][-1]["t"]: break
        if st["token"] == tok: st["playing"] = False; render()
    def play(_):
        if st["playing"]: return                         # ignore double clicks -> no overlapping animations
        if st["t"] >= cur()[3][-1]["t"]: st["t"] = 0
        st["playing"] = True; st["token"] += 1; render()
        threading.Thread(target=player, args=(st["token"],), daemon=True).start()
    def step(d):
        def f(_): pause(); st["t"] = max(0, min(cur()[3][-1]["t"], st["t"] + d)); render()
        return f
    def load(name, q): pause(); st.update(name=name, q=q, t=0); render()
    def on_act(c):
        st["loading"] = True; qd.options = ACT[c["new"]][3]; qd.value = qd.options[0]; st["loading"] = False; load(c["new"], qd.value)
    act.observe(on_act, names="value")
    qd.observe(lambda c: None if st["loading"] else load(st["name"], c["new"]), names="value")
    scrub.observe(lambda c: None if st["mute"] else (pause(), st.update(t=c["new"]), render()), names="value")
    bt["Play"].on_click(play); bt["Pause"].on_click(pause); bt["Next Step"].on_click(step(1))
    bt["Previous Step"].on_click(step(-1)); bt["Reset"].on_click(lambda _: (pause(), st.update(t=0), render()))

    comp = []                                            # comparison tab
    for name, (key, procs, cs, qs) in ACT.items():
        comp += [w.HTML(f"{CSS}<h3>{name}</h3><b>Comparison: Q = {qs[0]} ms vs Q = {qs[1]} ms</b>{table(comp_df(key, qs))}{conclusion(key, qs, cs)}")]
        for q in qs:
            S, df, _ = RUNS[(key, q)]
            comp += [w.Image(value=gantt(S, procs, S[-1]["t"], f"{name}, Q = {q} ms"), format="png", layout=w.Layout(width="95%")), w.HTML(table(df))]
    out = w.Output(); dl = w.Button(description="Download Gantt charts + tables (ZIP)", icon="download", button_style="primary", layout=w.Layout(width="330px"))
    def download(_):
        z = save_all()
        with out:
            out.clear_output(); print("Saved files in folder:", os.path.abspath(OUT))
            try: from google.colab import files; files.download(z)
            except Exception: print("Not in Colab - open the 'outputs' folder.")
    dl.on_click(download)
    live = w.VBox([w.HBox([act, qd]), w.HBox(list(bt.values())), speed, scrub, top, img, expl, ptab, mets])
    tabs = w.Tab(children=[live, w.VBox(comp), w.VBox([dl, out])])
    for i, n in enumerate(["Live Simulation", "Comparison & Conclusion", "Downloads"]): tabs.set_title(i, n)
    render(); return tabs

display(build_ui())